In [2]:
import math
import numpy as np

# https://www.ti.com/lit/an/slva882b/slva882b.pdf?ts=1746913705684
# https://www.ti.com/lit/an/slva477b/slva477b.pdf?ts=1746954692254&ref_url=https%253A%252F%252Fwww.bing.com%252F

In [3]:
Vin = 48 #V
Vout = 24 #V
Pmax = 850 #W
Iout = Pmax/Vout #A
print(Iout*1.25/2)
Istep = Iout

est_efficiency = 0.85

dVout_ripple = 0.2 #V
dVout_step = 2 #V
dVin = 1 #V
dIl = 0.25 #fraction of inductor DC current

fsw = 300*10**3 #Hz

phase_count = 2

duty_cycle = Vout/Vin
duty_cycle_adj = Vout/(Vin*est_efficiency)
print(duty_cycle_adj)

m = math.floor(phase_count*duty_cycle)
i_cin_norm_rms = math.sqrt((duty_cycle_adj-((m)/(phase_count)))*(((1+m)/(phase_count))-duty_cycle_adj))
print(i_cin_norm_rms) #overly idealized, must still expect some phase imbalance

i_cout_norm_rms = 0 # Overly optimistic, must still expect some

22.135416666666664
0.5882352941176471
0.1906100205414077


In [4]:
# Phase inductance
inductance_calc = (Vout * (1-duty_cycle))/(fsw*dIl*Iout/phase_count)
print(f"minimum phase inductance is {round(inductance_calc*10**6, 2)} uH") #uH
inductance_actual = 15*10**-6 #H

dIl_actual = (Vin-Vout)*duty_cycle_adj/(fsw*inductance_actual)


minimum phase inductance is 9.04 uH


In [5]:
# Cin

cin_phase = ((Iout/phase_count)*duty_cycle_adj*(1-duty_cycle_adj))/(fsw*dVin)
print(f"minimum cin per phase is {round(cin_phase*10**6, 2)} uF") #uF

minimum cin per phase is 14.3 uF


In [6]:
# Cout

cout_ripple = dIl_actual/(8*fsw*dVout_ripple)
print(f"minimum effective cout to address ripple is {round(cout_ripple*10**6,2)} uF")

t_undershoot = ((inductance_actual/phase_count)*Istep)/(Vin-Vout)
q_undershoot = (0.5*t_undershoot*Istep)
cout_undershoot = q_undershoot/dVout_step

t_overshoot = ((inductance_actual/phase_count)*Istep)/(Vout)
q_overshoot = 0.5*t_overshoot*Istep
cout_overshoot = q_overshoot/dVout_step

cout_actual = max(cout_ripple, cout_undershoot, cout_overshoot)
print(f"minimum total cout is {round(cout_actual*10**6, 2)} uF")

minimum effective cout to address ripple is 6.54 uF
minimum total cout is 98.0 uF


In [7]:
#vref calculator

def vref_calculator(vout, vref, rbot):
    rtop = rbot*(vout-vref)/vref
    return rtop

def vout_calculator(vref, rbot, rtop):
    vout = vref/(rbot/(rtop+rbot))
    return vout


In [8]:
# LV Buck FB

vref_buck = 0.8
rbot_buck = 1740*2

rtop = vref_calculator(Vout, vref_buck, rbot_buck)
rtop_actual_buck = 100000

vout_actual_buck = vout_calculator(vref_buck, rbot_buck, rtop_actual_buck)
print(vout_actual_buck)

23.788505747126436


In [ ]:
#DRVcc Calculations

Qg = 32*10**-9 #C
Vgate = 10 #V

Qtot = 4*Qg
print(Qtot)

gate_drive_current = Qtot*fsw
print(gate_drive_current*10**3) #mA

vout_linear = 11 #V
vref_linear = 1.173 #V
rbot_linear = 1300

rtop_linear = vref_calculator(vout_linear, vref_linear, rbot_linear)
print(rtop_linear)
rtop_actual_linear = 10000

vout_actual_linear = vout_calculator(vref_linear, rbot_linear, rtop_actual_linear)
print(vout_actual_linear)

p_dis_linear = gate_drive_current*(Vout-vout_actual_linear)
print(p_dis_linear)
t_rise_linear = 66.7*p_dis_linear
print(t_rise_linear)


1.28e-07
38.400000000000006
10890.963341858482
10.196076923076923
0.5300706461538462
35.355712098461545


0.418

In [137]:
# Equivalent Capacitance Function Definition
# https://www.power-mag.com/pdf/feature_pdf/1387888355_Murata_Feature_Layout_1.pdf

#Returned datastructure
class EquivalentCapacitance:
    def __init__(self, capacitances, fsw, ripple_current, equivalent_capacitance, equivalent_esr, ripple_voltage):
        self.capacitances = capacitances # Will be of the form [qty, capacitance (F), esr (Ohm), ripple current (A)] where the 4th column will only be added if a total ripple current was provided
        self.fsw = fsw
        self.ripple_current = ripple_current
        self.equivalent_capacitance = equivalent_capacitance
        self.equivalent_esr = equivalent_esr
        self.ripple_voltage = ripple_voltage

    def print(self):
        print(f"Capacitance Matrix: {self.capacitances}")
        print(f"Switching Frequency: {self.fsw} Hz")
        print(f"Ripple Current: {self.ripple_current} A")
        print(f"Equivalent Capacitance: {self.equivalent_capacitance} F")
        print(f"Equivalent ESR: {self.equivalent_esr} Ohms")
        print(f"Ripple Voltage: {self.ripple_voltage} V")

def equiv_capacitance(capacitances, fsw, current = 0):
    # capacitances is a 3 column, n row matrix with rows of the form [qty, capacitance (uF), esr (mOhm)]
    # fsw is the fundamental switching frequency (kHz)
    # current is optional. If provided, the return will include the ripple voltage and current per capacitor
    
    cs = np.array(capacitances)
    cs = cs.astype(float)
    xs = 1/(2*3.1416*fsw*cs[:, 1])
    
    cp = cs.copy()
    cp[:, 1] = cs[:, 1]/(1 + (cs[:, 2]/xs)**2)
    cp[:, 2] = (cs[:, 2]**2 + xs**2)/cs[:, 2]
    
    cpe = np.sum(cp[:, 0]*cp[:, 1])
    rpe = 1/np.sum(cp[:, 0]/cp[:, 2])

    xpe = 1/(2*3.1416*fsw*cpe)
    cse = cpe*(1 + (xpe/rpe)**2)
    rse = rpe/(1 + (rpe/xpe)**2)
    xse = 1/(2*3.1416*fsw*cse)

    dv = np.nan
    if(current != 0):
        dv = current*math.sqrt(rse**2 + xse**2)
        di = dv/np.sqrt(cs[:, 2]**2 + xs**2)
        cs = np.hstack((cs, di.reshape(-1, 1)))
    return EquivalentCapacitance(cs, fsw, current if current > 0 else np.nan, cse, rse, dv)

In [ ]:
# Effective Capacitance Calculations

actual_cout = np.array([[2, 0.07, 70], #100nF
                        [4, 4.4, 2.6], #10uF
                        [4, 1.2, 2.5], #4.7uF
                        [2, 47, 3390]]) #47u
actual_cout[:, 1] = actual_cout[:, 1]*10**-6 # Convert to F
actual_cout[:, 2] = actual_cout[:, 2]*10**-3 # Convert to Ohms
print(actual_cout)

capacitances = equiv_capacitance(actual_cout, fsw)
effective_capacitance = capacitances.equivalent_capacitance
effective_esr = capacitances.equivalent_esr
print(effective_capacitance)
print(effective_esr)

[[2.00e+00 7.00e-08 7.00e-02]
 [4.00e+00 4.40e-06 2.60e-03]
 [4.00e+00 1.20e-06 2.50e-03]
 [2.00e+00 4.70e-05 3.39e+00]]
2.2555752207249037e-05
0.0007523565608980738


In [192]:


# current_mode_compensation(effective_capacitance, effective_esr, vout_actual_buck/Iout, fsw, inductance_actual, duty_cycle_adj, 0.8, Vout, slope_compensation_factor)

current_mode_compensation(44*10**-6, 0.005, 3.3/3, 340000, 10*10**-6, 3.3/12, 1, 3.3)

34000.0
93414.52882614871
12374.948677966366
723429.8678148945
170000.0
0.05623413251903491


In [190]:
gm = 1.25*10**-3
R_comp = 5.911*10**3
20*math.log(R_comp*gm,10)

17.371419447138877